# Part 02: Collections and Loops

**Time needed:** 4-5 hours | **Prerequisite:** Part 01

Run each cell with **Shift+Enter**, read the output, then **change the code and run it again**.

Select your `.venv` kernel first (top right of the notebook).

## Why this part matters

In Part 01 every variable held **one** value. Real AI work is about **groups** of things:

| In AI engineering you handle... | ...which is stored as a |
|---|---|
| a conversation (many messages) | **list** |
| one message (`role` + `content`) | **dictionary** |
| token usage `(input, output)` | **tuple** |
| the unique labels in a dataset | **set** |
| "do this for every document" | **loop** |

By the end you'll be able to read the data an LLM API sends back and process a whole conversation with a few lines of code.

## What we'll cover

1. Lists
2. Tuples
3. Dictionaries
4. Sets
5. Loops: `for`, `while`, `range`, `enumerate`, `zip`
6. Comprehensions
7. Putting it together: trimming a chat history to fit a token budget

## 1. Lists

A **list** is an ordered collection of items, written in square brackets. Items can be added, removed and changed.

In [2]:
prompts = ["summaries", "translate", "classify"]

print(prompts)
print(type(prompts))
print(len(prompts))

['summaries', 'translate', 'classify']
<class 'list'>
3


### Indexing and slicing (just like strings!)

Positions start at **0**. Negative numbers count from the end. A slice `[start:stop]` goes up to, but **not including**, `stop`.

```
index:    0           1            2
        "summarise" "translate" "classify"
negative: -3          -2           -1
```

In [3]:
print(prompts[0])           # -------> This would print the first item
print(prompts[-1])          # -------> This would print the last item
print(prompts[0:2])         # -------> This would print items at the 0 and 1 position
print(prompts[1:])          # -------> This would print the item from index 1 to the end
print(prompts[:2])          # -------> This would print items from the start up to (not including) index 2



summaries
classify
['summaries', 'translate']
['translate', 'classify']
['summaries', 'translate']


### Changing a list

Lists are **mutable**: meaning that they can be changed in place. (Strings were *immutable*, remember?)

In [4]:
prompts[0] = "summarize"      # --------> replacing the item at index 0 from "summaries" to "summarize"
prompts.append("extract")     # --------> add to the END
prompts.insert(1, "rewrite")  # --------> This mean insert "rewrite" at position 1, shifting the rest along
print(prompts)

['summarize', 'rewrite', 'translate', 'classify', 'extract']


In [5]:
prompts.remove("classify")          # --------> Here we can remove an item by its VALUE (the first match)
print(prompts)

['summarize', 'rewrite', 'translate', 'extract']


In [6]:
last = prompts.pop()
print("popped:", last)
print(prompts)

popped: extract
['summarize', 'rewrite', 'translate']


### Useful list functions

Here is a list of token counts, one per request. These built-in functions work on any list of numbers:

In [7]:
token_counts = [120, 45, 300, 80]

print("how many   :", len(token_counts))
print("total      :", sum(token_counts))
print("smallest   :", min(token_counts))
print("largest    :", max(token_counts))
print("average    :", sum(token_counts) / len(token_counts))
print("is 45 in it:", 45 in token_counts)       # `in` works on lists too

how many   : 4
total      : 545
smallest   : 45
largest    : 300
average    : 136.25
is 45 in it: True


### Sorting: `sorted()` vs `.sort()`

- `sorted(x)` returns a **new** sorted list and leaves the original alone
- `x.sort()` sorts the list **in place** and returns `None`

In [8]:
token_counts = [120, 45, 300, 80]

print(sorted(token_counts))
print(sorted(token_counts, reverse=True))
print(token_counts)


[45, 80, 120, 300]
[300, 120, 80, 45]
[120, 45, 300, 80]


In [9]:
token_counts.sort()             # -------> .sort() Changes the list itself
print(token_counts)

[45, 80, 120, 300]


> **Classic bug:** `token_counts = token_counts.sort()` sets the variable to `None`, because `.sort()` returns nothing. Either call `.sort()` on its own line or use `sorted()`.

### Lists can hold anything, even other lists

An **embedding** (a way to turn text into numbers so a model can compare meanings) is just a list of floats. Real ones have hundreds or thousands of numbers; here are three:

In [10]:
embedding = [0.12, -0.45, 0.88]
print(len(embedding), "dimensions")

3 dimensions


In [11]:
# A list of lists: one embedding per document
document_embeddings = [
    [0.12, -0.45, 0.88],
    [0.50, 0.10, -0.20],
]
print(document_embeddings[1])               # ----------> this prints the second embedding
print(document_embeddings[1][0])            # ----------> this prints the first number of the second embedding

[0.5, 0.1, -0.2]
0.5


### Gotcha: copying a list

`b = a` does **not** copy a list. Both names point at the **same** list:

In [13]:
original = ["a", "b"]
alias = original                # NOT a copy: it's just a second name for the same list

alias.append("c")
print(alias)
print(original)                 # It's surprising to see "original changes too!"


['a', 'b', 'c']
['a', 'b', 'c']


In [15]:
safe = original.copy() # a real, independent copy
safe.append("d")
print(original)         # unchanges by the copy
print(safe)

['a', 'b', 'c']
['a', 'b', 'c', 'd']


---
## 2. Tuples

A **tuple** is like a list that **cannot be changed** after creation. It uses round brackets. Use one when the data is a fixed bundle: *"these two numbers always go together."*

APIs often report token usage this way: `(input_tokens, output_tokens)`.

In [16]:
usage = (1200, 350)

print(usage)
print(type(usage))
print(usage[0])                 # Indexing works same way with list
print(usage[-1])
print(len(usage))

(1200, 350)
<class 'tuple'>
1200
350
2


Trying to change one raises an error:

```python
usage[0] = 999     # TypeError: 'tuple' object does not support item assignment
```

### Unpacking: the best thing about tuples

You can split a tuple into separate variables in one line. The number of names must match the number of items:

In [17]:
usage = (1200, 350)

input_tokens, output_tokens = usage                 # first item -> first name, second -> second name
print("input :", input_tokens)
print("output:", output_tokens)
print("total :", input_tokens + output_tokens)

# swapping two variables is a one-liner. All thanks to tuples
a, b = 1, 2
a, b = b, a
print(a, b)

input : 1200
output: 350
total : 1550
2 1


### Gotcha: a one-item tuple needs a trailing comma

Round brackets alone are just maths grouping:


In [18]:
print(type((5)))            # int: just the number 5 in brackets
print(type((5,)))           # tuple: the comma makes it one

<class 'int'>
<class 'tuple'>


### List of tuples: a very common shape

You'll often loop over pairs like this. Notice we unpack each pair directly in the `for` line:

In [19]:
word_counts = [("token", 4), ("cost", 3), ("model", 2)]

for word, count in word_counts:
    print(f"{word} appears {count} times")

token appears 4 times
cost appears 3 times
model appears 2 times
